# Exercise 5.4 — Random forests and tuning with a workflow (with solutions)

*HDS MSc · Module 3 · the Python version of this exercise; the web page is at <https://inflamuob.github.io/HDS_M3/day5/day5-04-random-forest.html>.*

*Open it in Jupyter, VS Code or Positron and run it a cell at a time. Keep it in the folder that holds your `data` folder. This version has the worked answers filled in.*

A single decision tree is easy to read but unstable: a few different people in
the training rows can give a very different tree, and a deep tree overfits.
A **random forest** grows hundreds of trees, each on a different bootstrap
sample of the training rows and each allowed to look at only a random subset of
the predictors at every split, and averages their votes. The averaging cancels
much of the noise that each single tree has memorised.

**Goal.** In this exercise you will:

- fit a random forest classifier to the diabetes data;
- tune its key hyperparameters with a workflow (R) or a pipeline search
  (Python);
- compare training and test performance;
- reflect on what workflows and pipelines give you.

**Theory:** bagging and random predictor subsets; the hyperparameters `mtry`
and `min_n`; grid search with cross-validation; finalising a tuned model; why a
forest's training performance says little.

> **The plots are for looking at**
>
> The code for every plot is included, so that you can run it and see the figure. You are not expected to write it yourself.

### Packages and data

In [ ]:
# Packages for this exercise. If one is missing, install it once from a terminal
# (in a Jupyter notebook, type %pip instead of pip):
# pip install pandas numpy matplotlib scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.metrics import roc_auc_score, classification_report

The same 750 people and split as Exercises 5.2 and 5.3, from
[`data/diabetes_clean.csv`](data/diabetes_clean.csv), with the five predictors pregnant,
glucose, mass, pedigree and age (triceps, insulin and blood pressure are left
out because a zero there means "not measured"). The outcome is 1 for `pos` and
0 for `neg`.

In [ ]:
diabetes = pd.read_csv("data/diabetes_clean.csv")
diabetes = diabetes[["pregnant", "glucose", "mass", "pedigree", "age", "diabetes"]]
diabetes["diabetes"] = (diabetes["diabetes"] == "pos").astype(int)       # 1 = pos, 0 = neg

is_test = diabetes.index % 5 == 0               # every fifth person is held back
train = diabetes[~is_test]
test  = diabetes[is_test]

predictors = ["pregnant", "glucose", "mass", "pedigree", "age"]

------------------------------------------------------------------------

### Part 1 — A random forest with tunable hyperparameters

`RandomForestClassifier()`, with the number of trees fixed at 500,
`random_state` fixed so that the forest is the same every time, and `n_jobs=-1`
to use all the computer's cores. Two
hyperparameters will be chosen by the search:

- `max_features`: the number of predictors drawn at random as candidates at each
  split (1 to 5 here), called `mtry` in R;
- `min_samples_split`: the minimum number of people a node must hold to be
  split, called `min_n` in R.

In [ ]:
rf = RandomForestClassifier(n_estimators=500, random_state=123, n_jobs=-1)
print(rf)

> **Learning points**
>
> - A random forest averages many trees, each grown on a bootstrap sample with a random subset of predictors at every split.
> - `mtry` (`max_features`) controls how different the trees are from each other; `min_n` (`min_samples_split`) how far each tree may grow.

------------------------------------------------------------------------

### Part 2 — The grid, the folds and the search

Five values of `max_features` and five of `min_samples_split`, 25 combinations:

In [ ]:
rf_grid = {"max_features":      [1, 2, 3, 4, 5],
           "min_samples_split": [2, 4, 6, 8, 10]}

Each combination is judged by 5-fold cross-validation on the training rows, as
in Exercise 5.3: 25 combinations times 5 folds is 125 forests of 500 trees.
The forest was created with `n_jobs=-1`, which grows its trees on all the
computer's cores at once and makes the search several times faster.

In [ ]:
rf_search = GridSearchCV(rf, param_grid=rf_grid, cv=5, scoring="roc_auc")
rf_search = rf_search.fit(train[predictors], train["diabetes"])

rf_results = pd.DataFrame(rf_search.cv_results_)
rf_results = rf_results[["param_max_features", "param_min_samples_split",
                         "mean_test_score", "std_test_score"]]
print(rf_results.round(3).to_string())

> **Learning points**
>
> - A grid search fits one model per combination and fold; keep the grid small and the number of trees modest.
> - Cross-validation uses the training rows only; the test rows stay untouched.

------------------------------------------------------------------------

### Part 3 — Inspect the tuning results

Plot the cross-validated AUC of every combination:

In [ ]:
fig, ax = plt.subplots()
for split in rf_grid["min_samples_split"]:
    rows = rf_results[rf_results["param_min_samples_split"] == split]
    ax.plot(rows["param_max_features"], rows["mean_test_score"], marker="o", label=str(split))
ax.set_xlabel("max_features (mtry)")
ax.set_ylabel("Mean cross-validated AUC")
ax.legend(title="min_samples_split")
plt.show()

Now list the five best combinations by AUC:

> **Worked answer**

In [ ]:
rf_results.sort_values("mean_test_score", ascending=False).head(5).round(3)

> R: `mtry` 1 with `min_n` 10 (AUC 0.783), 2 with 10 (0.781), 1 with 8 (0.779),
> 1 with 6 (0.778), 2 with 8 (0.777), each with a standard error of about 0.012.
> Python: `max_features` 1 with `min_samples_split` 10 (0.786), 1 with 8 (0.784),
> 2 with 10 (0.781), 2 with 8 (0.778), 1 with 6 (0.778). Python's
> `std_test_score` is the standard deviation across the 5 folds, about 0.039;
> divided by $\sqrt{5}$ it gives a standard error of about 0.018. In both
> languages the smallest `mtry` and the largest `min_n` win, and the best `min_n`
> lies at the edge of the grid.

**Question!** Which combinations of `mtry` and `min_n` seem to work best? Do
more flexible forests (more predictors per split, smaller nodes) always perform
better? How large are the differences compared with the standard errors?

> **Learning points**
>
> - Look at the whole grid, not only the winner: here many combinations are within one standard error of the best.
> - Small `mtry` makes the trees more different from one another, which is what makes averaging them work.

------------------------------------------------------------------------

### Part 4 — The final forest, evaluated once

The best combination by AUC. `GridSearchCV()` has already refitted a forest with
it on all 600 training rows (`refit=True` is the default):

In [ ]:
print(rf_search.best_params_)
print("cross-validated AUC:", round(rf_search.best_score_, 3))

final_rf = rf_search.best_estimator_

Predict both sets and compute the metrics, starting with the training rows:

In [ ]:
train_preds = train.copy()
train_preds["prob"] = final_rf.predict_proba(train[predictors])[:, 1]
train_preds["pred_class"] = final_rf.predict(train[predictors])

test_preds = test.copy()
test_preds["prob"] = final_rf.predict_proba(test[predictors])[:, 1]
test_preds["pred_class"] = final_rf.predict(test[predictors])

truth, predicted = train_preds["diabetes"], train_preds["pred_class"]
print("train accuracy:   ", round(accuracy_score(truth, predicted), 3))
print("train precision:  ", round(precision_score(truth, predicted), 3))
print("train sensitivity:", round(recall_score(truth, predicted), 3))
print("train specificity:", round(recall_score(truth, predicted, pos_label=0), 3))
print("train AUC:        ", round(roc_auc_score(truth, train_preds["prob"]), 3))

Now the test rows:

> **Worked answer**

In [ ]:
truth, predicted = test_preds["diabetes"], test_preds["pred_class"]
print("test accuracy:   ", round(accuracy_score(truth, predicted), 3))
print("test precision:  ", round(precision_score(truth, predicted), 3))
print("test sensitivity:", round(recall_score(truth, predicted), 3))
print("test specificity:", round(recall_score(truth, predicted, pos_label=0), 3))
print("test AUC:        ", round(roc_auc_score(truth, test_preds["prob"]), 3))

> R: test accuracy 0.767, precision 0.725, sensitivity 0.547, specificity 0.887,
> AUC 0.825 (training accuracy 0.912, AUC 0.982). Python: 0.780, 0.750, 0.566,
> 0.897, AUC 0.825 (training accuracy 0.895, AUC 0.970).

Forests are random (bootstrap samples, predictor subsets, fold assignment), and
R and Python draw different random numbers, so the two languages do not give
the same forest or exactly the same numbers.

**Question!** Is there still a big gap between training and test performance?
A forest predicts its own training rows with trees that have seen most of them,
so its training performance is always flattering: judge it on the test rows or
by cross-validation.

> **Learning points**
>
> - Finalise: put the best hyperparameters into the workflow and fit once on all the training rows.
> - A forest's training performance is close to perfect by construction; its test or cross-validated performance is the one to report.
> - The test rows are used once, at the end.

------------------------------------------------------------------------

### Part 5 — Extra: was the tuning worth it?

Fit a basic forest with scikit-learn's default hyperparameters (100 trees, nodes
split down to 2 people, `max_features` the square root of the number of
predictors: 2) and compare it on the test rows with the tuned forest.
`classification_report()` prints the precision and recall of each class in one
table; the recall of class 0 is the specificity.

In [ ]:
rf_basic = RandomForestClassifier(random_state=123)
rf_basic = rf_basic.fit(train[predictors], train["diabetes"])

basic_class = rf_basic.predict(test[predictors])
basic_prob  = rf_basic.predict_proba(test[predictors])[:, 1]

print("basic forest, test accuracy:", round(accuracy_score(test["diabetes"], basic_class), 3))
print("basic forest, test AUC:     ", round(roc_auc_score(test["diabetes"], basic_prob), 3))
print(classification_report(test["diabetes"], basic_class, digits=3))

The untuned forest does as well as the tuned one on the test rows: test AUC
0.832 against 0.825 in R, and 0.835 against 0.825 in Python (test accuracy 0.773
against 0.767 in R, 0.727 against 0.780 in Python). The cross-validated AUCs of
the whole grid lay within about 0.02 of each other, one or two standard errors,
and 150 test patients cannot tell differences that small apart. Tuning is still
the right procedure; here it simply has little to find.

**Question! Why pipelines?** Look back at the code of this exercise and of
Exercise 5.3.

- What do workflows (R) and pipelines (Python) give you, compared with writing
  the model, the formula, the pre-processing and the predictions separately by
  hand?
- How does a workflow help when you change the model, say from a tree to a
  random forest to a logistic regression? Which lines changed between
  Exercise 5.3 and this one?
- How does it help when you add pre-processing steps such as scaling,
  imputation or dummy variables, especially during cross-validation, where those
  steps must be learned again on each fold's training part?

> **Learning points**
>
> - Compare a tuned model with the untuned default: tuning costs time and does not always pay.
> - A workflow or pipeline keeps pre-processing and model together, so changing either is one line, and cross-validation repeats both on every fold.

## Stretch

> **Stretch — the forest against the logistic regression**
>
> Exercise 5.2 fitted a logistic regression on the same five predictors and the
> same split. Put its test AUC next to the tuned forest's. Which model would you
> take to a clinic, and what besides the AUC would you weigh?